# TEI-XML: Encoding Is Interpretation

**Computational Text Analysis Pack · Module 20 — Encoding as Method**

*Technique:* TEI P5 structure; markup as a scholarly claim rather than formatting

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/adh-dh-notebooks/blob/main/09-encoding-as-method/20a_tei_and_what_markup_asserts.ipynb)

Data: `colonial_texts.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
from collections import Counter

# The data this notebook needs. It is downloaded from CourseHub automatically;
# uploading by hand is only a fallback if that server is unreachable.
NEEDED = ['colonial_texts.csv']
DATA_URL = 'https://coursehub.africandigitalhumanities.ca/ide/pack-data/text/'

def _missing():
    return [f for f in NEEDED if not os.path.exists(f)]

missing = _missing()
if missing:
    # Fetch from CourseHub first, so nobody has to upload anything.
    import urllib.request, urllib.parse
    for _f in list(missing):
        try:
            urllib.request.urlretrieve(DATA_URL + urllib.parse.quote(_f), _f)
            print(f'downloaded {_f}')
        except Exception as _e:
            print(f'could not fetch {_f}: {_e}')
    missing = _missing()

if missing:
    try:
        from google.colab import files
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))
    # Ask again until everything has arrived. The upload widget returns as soon
    # as you close it, so picking only some of the files would otherwise fail a
    # few lines below with a confusing FileNotFoundError.
    for _ in range(4):
        print('Select ALL of these at once (ctrl-click / cmd-click to multi-select):')
        print('   ' + ', '.join(missing))
        files.upload()
        missing = _missing()
        if not missing:
            break
        print('Still needed: ' + ', '.join(missing))
    if missing:
        raise SystemExit(
            'Missing: ' + ', '.join(missing) + '. Re-run this cell and select '
            'every file listed, or upload them with the folder icon on the left.')

df_colonial = pd.read_csv('colonial_texts.csv')
NIGERIAN = ['Hausa', 'Igbo', 'Yoruba', 'Nigerian Pidgin']
# Most of this pack runs on scikit-learn and the standard library. Also
# available: nltk, spacy, gensim and networkx (modules 14-19), Keras
# (case studies), and a GPU pod reached through /ide/api/neural/ for the
# neural modules — never called directly, so no token lives in a notebook.

print(f'Loaded. df_colonial has {len(df_colonial):,} rows and {df_colonial.shape[1]} columns.')

## What this notebook does

Every corpus in this course arrived as a CSV with a `text` column. That format
already contains decisions — someone decided where a document starts and stops,
what counted as a passage, which front matter to discard — and the CSV hides all
of them.

**TEI-XML** is the humanities standard for making those decisions explicit and
arguable. The survey's formulation is the one to hold onto: *encoding is
interpretation*. A tag is not formatting. It is a claim that this span of
characters is a person, a date, a correction, a passage in another hand.

## The tooling

**TEI P5** is the Text Encoding Initiative's current guidelines, revised roughly
twice a year. Python's standard library ships `xml.etree.ElementTree`, which is
what we use here; `lxml` is faster and supports full XPath, and **TEI Publisher**
turns TEI into a reading interface. Querying and transforming TEI is done with
**XPath**, **XQuery** and **XSLT** — the next notebook is XPath.

Nothing here needs a special library, which is part of the argument: TEI is a
vocabulary and a set of scholarly commitments, not a piece of software.

In [ ]:
import re

def tokenise(text):
    """Words, lowercased, Unicode-aware.

    `\\w` matches Unicode letters, so ọ, à, ƙ, ɗ and ɓ survive. It does not
    match combining marks, and a tone mark on ọ or ẹ is always one, so the
    pattern adds the combining-accent range U+0300-U+036F; without it ọ̀rọ̀
    splits into ọ + rọ. The common shortcut `[a-z]+` throws away far more and
    quietly changes every count in this notebook — see notebook 03b.
    """
    return re.findall(r"[\w\u0300-\u036f]+", str(text).lower())

sample = "Àwọn ọmọ ìlú ti ṣe ìpàdé lórí ọ̀rọ̀ ẹ̀kọ́ — ƙungiyar ɗalibai ta yi taro"
print("sample     :", sample)
print("tokens     :", tokenise(sample))
print("bare \\w+   :", re.findall(r"\w+", sample.lower()))
print("naive [a-z]:", re.findall(r"[a-z]+", sample.lower()))
print()
print("The naive pattern splits words at every accented letter; bare \\w+")
print("splits them wherever a tone mark sits on ọ or ẹ.")

In [ ]:
import xml.etree.ElementTree as ET
import re
from xml.sax.saxutils import escape

TEI_NS = "http://www.tei-c.org/ns/1.0"

passage = df_colonial.iloc[3]
raw = str(passage["text"])[:700]
print(f"work : {passage['title'][:60]}")
print(f"author: {passage['author']}")
print()
print(raw[:400], "...")

## Encoding a passage, decision by decision

Below, each markup decision is written as a separate step so you can see that it
is a decision. A different scholar would make different ones, and the resulting
file would support different research.

In [ ]:
def encode_tei(text, title, author, work_id):
    body = escape(text)
    # Decision 1: personal names. We use the module 15c rule — and inherit its
    # bias toward capitalised European names.
    body = re.sub(r"\b((?:Mr|Mrs|Dr|Sir|Lord|Captain|Chief|King)\.?\s+[A-Z]\w+)",
                  r"<persName>\1</persName>", body)
    # Decision 2: dates. Only four-digit years, only 1600-1999.
    body = re.sub(r"\b(1[6-9]\d{2})\b", r'<date when="\1">\1</date>', body)
    # Decision 3: places, after a locative preposition. Weak, and stated as such.
    body = re.sub(r"\b(?:at|in|near)\s+([A-Z][a-z]{3,})",
                  r'<placeName>\1</placeName>', body)
    # Decision 4: paragraphs are blank-line separated. The source had no marks.
    paras = "\n      ".join(f"<p>{p.strip()}</p>"
                            for p in re.split(r"\n\s*\n|(?<=\.)\s{2,}", body)
                            if p.strip())
    return f'''<TEI xmlns="{TEI_NS}">
  <teiHeader>
    <fileDesc>
      <titleStmt><title>{escape(title)}</title>
        <author>{escape(str(author))}</author></titleStmt>
      <publicationStmt><p>Public domain, via Project Gutenberg.</p></publicationStmt>
      <sourceDesc><p>Passage {work_id} of the ADH colonial-discourse corpus.</p></sourceDesc>
    </fileDesc>
    <encodingDesc>
      <p>Names, dates and places tagged automatically by regular expression.
         Coverage is partial and biased toward English orthography.</p>
    </encodingDesc>
  </teiHeader>
  <text><body>
      {paras}
  </body></text>
</TEI>'''

tei = encode_tei(raw, passage["title"], passage["author"], passage["work_id"])
print(tei[:1400])

## The header is the scholarship

Notice how much of that file is not the text. The `<teiHeader>` carries the
title statement, the publication and source description, and — the part that
matters most for reuse — an `<encodingDesc>` saying **how** the markup was
produced and how far to trust it.

That paragraph is the difference between a dataset someone else can use and one
they can only guess at. Our header admits the names were tagged by regex and
that coverage is biased. A CSV has nowhere to put that admission.

In [ ]:
root = ET.fromstring(tei)
ns = {"t": TEI_NS}

print("What the header asserts:")
for path, label in [("t:teiHeader//t:title", "title"),
                    ("t:teiHeader//t:author", "author"),
                    ("t:teiHeader//t:encodingDesc/t:p", "encoding method")]:
    el = root.find(path, ns)
    if el is not None and el.text:
        print(f"  {label:<16}{' '.join(el.text.split())[:80]}")

print("\nWhat the body asserts:")
for tag in ["p", "persName", "date", "placeName"]:
    found = root.findall(f".//t:{tag}", ns)
    print(f"  <{tag}>{'':<12}{len(found):>3} elements", end="")
    if found and tag != "p":
        print("   e.g. " + ", ".join(f.text for f in found[:4] if f.text))
    else:
        print()

## Where the interpretation shows

Two encoders given the same page produce different files. The disagreements are
not errors to be eliminated — they are the scholarly content of the edition, and
they are exactly what module 17's agreement statistics were built to measure.

In [ ]:
# The same passage under a second, more inclusive naming policy.
def encode_alt(text):
    body = escape(text)
    # This encoder tags every capitalised word not at a sentence start.
    body = re.sub(r"(?<![.!?]\s)(?<!^)\b([A-Z][a-z]{2,})\b",
                  r"<persName>\1</persName>", body)
    return body

a_names = {e.text for e in root.findall(".//t:persName", ns) if e.text}
alt = re.findall(r"<persName>([^<]+)</persName>", encode_alt(raw))
b_names = set(alt)

print(f"encoder A (title-based rule) tagged {len(a_names)}: {sorted(a_names)}")
print(f"encoder B (capital-based rule) tagged {len(b_names)}")
print()
print(f"agreed on      : {sorted(a_names & b_names)}")
print(f"only encoder B : {sorted(b_names - a_names)[:10]}")
print()
print("Encoder B catches names without titles and also sweeps in place names,")
print("months and ship names. Neither file is 'the text'. Each is an argument")
print("about what the text contains, and TEI's job is to make that argument")
print("explicit enough to disagree with.")

### 🔧 Your turn 1

Add a `<foreign>` tag for words you judge to be non-English, and encode a passage. How confident are you in each decision?

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

Most people find the boundary cases immediately: a word that entered English
through colonial contact — *veldt*, *kraal*, *bazaar* — is foreign in one
scholar's judgement and naturalised in another's. There is no lookup that
settles it, which is why TEI provides the tag rather than the answer. Encoding
it forces the question into the open and records your answer where a reader can
contest it; leaving the text as plain characters merely hides that the question
was ever asked.

</details>